# NB10 - Web app (FastAPI backend + browser frontend)
Starts the backend inside Colab and shows the frontend in the notebook (or in a new tab). Upload an OASIS-1 `*_masked_gfc.hdr` + `.img` pair,
or run a cached participant. The page shows the prediction **or the abstention and its reasons**, probabilities, conformal set, quality and OOD checks, the age/sex-only comparison and Grad-CAM.
It is a research demonstration - not a medical device.

In [ ]:
# ── 1. Environment Detection (Google Colab vs Local Machine) ──────────────
import sys, os, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── 2. Configure Paths ───────────────────────────────────────────────────
if IN_COLAB:
    # Google Colab paths (on your Google Drive)
    PROJECT  = '/content/drive/MyDrive/oasis_project'      # this unzipped project folder on Drive
    DATA_DIR = '/content/drive/MyDrive/OASIS1'             # folder with the 12 OASIS-1 disc archives (+ clinical csv)
    OUT_DIR  = '/content/drive/MyDrive/OASIS1_outputs'     # results are saved here (survives Colab disconnects)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PROJECT}/requirements.txt'], check=False)
else:
    # Local machine paths (runs directly on your laptop)
    PROJECT  = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    DATA_DIR = os.path.join(PROJECT, 'Dataset')
    OUT_DIR  = os.path.join(PROJECT, 'OASIS1_outputs')

# ── 3. install + load ─────────────────────────────────────────────────────
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
from oasis_ad.config import load_config, Paths
from oasis_ad import pipeline as PL, DISCLAIMER
cfg = load_config(os.path.join(PROJECT, 'configs', 'default.yaml'),
                  {'paths.data_dir': DATA_DIR, 'paths.out_dir': OUT_DIR})
paths = Paths(cfg)
import torch, pandas as pd
print(DISCLAIMER)
print('Environment:', 'Google Colab' if IN_COLAB else 'Local Laptop')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (on Colab switch to Runtime > Change runtime type > T4 GPU)')

In [ ]:
from oasis_ad.app.server import run_in_thread
from oasis_ad.data.preprocess import sync_cache_to_local
sync_cache_to_local(paths)
server = run_in_thread(paths.bundle, port=cfg.app.port, cache_dir=paths.cache_local,
                       manifest_csv=f'{paths.cache_drive}/manifest_qc_passed.csv')
print('backend running on port', cfg.app.port)

In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(cfg.app.port, height=950)
    print("If embedded view doesn't load, use: output.serve_kernel_port_as_window(cfg.app.port)")
except ImportError:
    print(f"\n>>> App running! Open http://localhost:{cfg.app.port} in your browser to view the interface <<<\n")

In [ ]:
# API check (backend side)
import requests; print(requests.get(f'http://localhost:{cfg.app.port}/api/model-info').json())